# 可选：一个已经微调好的中文情感模型

请使用 PyTorch 环境。这个检查点是在**京东商品评论**二分类任务上微调得到的，
不是在语言模型上临时接一个未训练的Classification Head；它的公开分数来自京东任务，
不能替代本课在酒店/影视评论上的 LSTM 实验——领域不同，指标不能互相顶替。
首次运行会下载模型文件。


In [1]:
import sys
from importlib.metadata import version
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
assert sys.version_info[:2] == (3, 12), "请选择项目的 Python 3.12 内核。"
assert version('torch').split("+")[0] == '2.14.1', "请选择项目锁定的运行环境。"
print("Python：", sys.version.split()[0])
print('torch' + "：", version('torch'))
print("运行模式：", '课堂默认训练参数')

Python： 3.12.12
torch： 2.14.1
运行模式： 课堂默认训练参数


**预期输出：**

```text
Python： 3.12.12
torch： 2.14.1
运行模式： 课堂默认训练参数
```

In [2]:
from pathlib import Path
import os
import sys

# 不依赖当前工作目录，向上找到中文课程根目录。
start = Path.cwd().resolve()
COURSE = next((p for p in [start, *start.parents]
               if (p / "CHINESE_COURSE_ROOT").is_file()), None)
if COURSE is None:
    raise RuntimeError("请在项目根目录（中文课程）中启动 Jupyter。")
ENGLISH = COURSE / "english"
# 中文专用模块在仓库根 common/，与英文课程共享的数据与检测模块在 english/common。
sys.path.insert(0, str(ENGLISH / "common"))
sys.path.insert(0, str(COURSE / "common"))
# 模型权重缓存与英文课程共用一份，避免同一份权重重复下载。
os.environ.setdefault("HF_HOME", str(ENGLISH / "data" / "huggingface"))
os.environ.setdefault("TORCH_HOME", str(ENGLISH / "data" / "torch-cache"))

'/Users/will/workspace/deep-learning-lab/english/data/torch-cache'

**预期输出：**

```text
'…/deep-learning-lab/english/data/torch-cache'
```

（输出里的 `…` 代表运行本 notebook 时所在机器的仓库绝对路径前缀，你在自己机器上执行会看到自己的路径。）

In [3]:
from transformers import pipeline
import torch
model_id = "uer/roberta-base-finetuned-jd-binary-chinese"
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")
classifier = pipeline("sentiment-analysis", model=model_id, device=DEVICE)
LABEL_CN = {"negative": "负面", "positive": "正面", "0": "负面", "1": "正面",
            "LABEL_0": "负面", "LABEL_1": "正面",
            "positive (stars 4 and 5)": "正面", "negative (stars 1, 2 and 3)": "负面"}
reviews = ["房间干净，位置方便，前台服务也很热情。",
           "隔音很差，房间又小，下次不会再来住了。"]
for text, result in zip(reviews, classifier(reviews, truncation=True, max_length=512)):
    label = LABEL_CN.get(result["label"].lower(), result["label"])
    print(f"{label}（{result['score']:.3f}）：{text}")

/Users/will/env_dl_uv/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device set to use mps


正面（0.994）：房间干净，位置方便，前台服务也很热情。
负面（0.977）：隔音很差，房间又小，下次不会再来住了。


**预期输出：**

```text
/Users/will/env_dl_uv/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use mps
正面（0.994）：房间干净，位置方便，前台服务也很热情。
负面（0.977）：隔音很差，房间又小，下次不会再来住了。
```

## 讨论

评论长度超过模型上限时会发生什么？京东商品评论与本课的酒店/影视评论有什么差别？
为什么置信度很高也可能预测错？

[模型主页与来源说明](https://huggingface.co/uer/roberta-base-finetuned-jd-binary-chinese)。
